In [1]:
import os
import re
import time
import random
from datetime import date, datetime, timedelta
from typing import Dict, Iterator, List, Optional, Tuple

import requests
import pandas as pd


# =========================
# CONFIG
# =========================
os.environ["GOVINFO_API_KEY"] = "u9B8x7eCnRqZMd2pm0pfUSw7LDATCpveLDIzTJdN"

API_KEY = os.getenv("GOVINFO_API_KEY", "REPLACE_ME")
BASE_URL = "https://api.govinfo.gov"

# GovInfo docs: pageSize is limited to 1000 and offsetMark should start as "*". :contentReference[oaicite:4]{index=4}
PAGE_SIZE = 1000

# throttle between page fetches
THROTTLE_SECONDS = 0.8

# House hearings in CHRG packageIds look like: CHRG-116hhrg36909 (lowercase hhrg is common)
HOUSE_PID_RE = re.compile(r"^CHRG-\d+hhrg", re.IGNORECASE)
JOINT_PID_RE = re.compile(r"^CHRG-\d+jhrg", re.IGNORECASE)
SENATE_PID_RE = re.compile(r"^CHRG-\d+shrg", re.IGNORECASE)


# =========================
# HTTP RETRY
# =========================

def get_page(
    session: requests.Session,
    url: str,
    params: Dict,
    max_tries: int = 12,
    timeout: int = 60,
) -> Dict:
    """
    Exponential backoff + jitter; retries 429 and 5xx.
    """
    if API_KEY == "REPLACE_ME":
        raise ValueError("API key not set. Set GOVINFO_API_KEY env var or set API_KEY directly.")

    last_status = None
    for i in range(1, max_tries + 1):
        resp = session.get(url, params=params, timeout=timeout)
        last_status = resp.status_code

        if last_status == 200:
            return resp.json()

        if last_status == 429 or (500 <= last_status < 600):
            base_wait = min(300, 2 ** (i - 1))
            wait = base_wait + random.random()
            print(f"HTTP {last_status} (try {i}/{max_tries}). Sleeping {wait:.1f}s then retrying...")
            time.sleep(wait)
            continue

        resp.raise_for_status()

    raise RuntimeError(f"Failed after {max_tries} tries. Last status: {last_status}")


# =========================
# DATE SLICERS
# =========================

def last_day_of_month(y: int, m: int) -> date:
    if m == 12:
        return date(y, 12, 31)
    return date(y, m + 1, 1) - timedelta(days=1)

def month_slices(year: int) -> Iterator[Tuple[str, str, str]]:
    for m in range(1, 13):
        start = date(year, m, 1)
        end = last_day_of_month(year, m)
        yield start.isoformat(), end.isoformat(), f"{year}-{m:02d}"

def quarter_slices(year: int) -> Iterator[Tuple[str, str, str]]:
    quarters = [(1, 3, "Q1"), (4, 6, "Q2"), (7, 9, "Q3"), (10, 12, "Q4")]
    for sm, em, label in quarters:
        start = date(year, sm, 1)
        end = last_day_of_month(year, em)
        yield start.isoformat(), end.isoformat(), f"{year}-{label}"


# =========================
# PAGINATORS
# =========================

def iter_published_packages(
    session: requests.Session,
    date_issued_start: str,
    date_issued_end: str,
    collection: str = "CHRG",
    throttle_seconds: float = THROTTLE_SECONDS,
    page_size: int = PAGE_SIZE,
) -> Iterator[Dict]:
    """
    /published/{start}/{end} discovers packages by official publication date (dateIssued). :contentReference[oaicite:5]{index=5}
    Pagination: offsetMark="*" then follow nextPage. :contentReference[oaicite:6]{index=6}
    """
    url = f"{BASE_URL}/published/{date_issued_start}/{date_issued_end}"
    offset_mark = "*"
    page = 1

    while True:
        params = {
            "collection": collection,
            "pageSize": page_size,
            "offsetMark": offset_mark,
            "api_key": API_KEY,
        }

        print(f"[published] {date_issued_start}..{date_issued_end} | page {page} | offsetMark={offset_mark}")
        data = get_page(session, url, params)

        packages = data.get("packages") or []
        for p in packages:
            yield p

        next_off = data.get("nextPage") or ""
        if not next_off:
            break

        offset_mark = next_off
        page += 1
        time.sleep(throttle_seconds)


def iter_collection_modified_packages(
    session: requests.Session,
    collection: str,
    last_modified_start: str,
    last_modified_end: Optional[str] = None,
    throttle_seconds: float = THROTTLE_SECONDS,
    page_size: int = PAGE_SIZE,
) -> Iterator[Dict]:
    """
    /collections/{collection}/{start}/{end?} retrieves packages added/modified in a time window. :contentReference[oaicite:7]{index=7}
    Uses same offsetMark pagination. :contentReference[oaicite:8]{index=8}

    Dates are timestamps in ISO-like formats accepted by GovInfo (commonly YYYY-MM-DD or YYYY-MM-DDTHH:MM:SSZ).
    """
    if last_modified_end:
        url = f"{BASE_URL}/collections/{collection}/{last_modified_start}/{last_modified_end}"
    else:
        url = f"{BASE_URL}/collections/{collection}/{last_modified_start}"

    offset_mark = "*"
    page = 1

    while True:
        params = {
            "pageSize": page_size,
            "offsetMark": offset_mark,
            "api_key": API_KEY,
        }

        print(f"[collections] {collection} {last_modified_start}..{last_modified_end or ''} | page {page} | offsetMark={offset_mark}")
        data = get_page(session, url, params)

        packages = data.get("packages") or []
        for p in packages:
            yield p

        next_off = data.get("nextPage") or ""
        if not next_off:
            break

        offset_mark = next_off
        page += 1
        time.sleep(throttle_seconds)


# =========================
# NORMALIZATION + FILTERS
# =========================

def packages_to_df(packages: List[Dict], extra: Optional[Dict] = None) -> pd.DataFrame:
    extra = extra or {}
    rows = []
    for p in packages:
        rows.append({
            "packageId": p.get("packageId"),
            "title": p.get("title"),
            "congress": pd.to_numeric(p.get("congress"), errors="coerce"),
            "dateIssued": p.get("dateIssued"),
            "chamber": p.get("chamber"),
            "docClass": p.get("docClass"),
            "lastModified": p.get("lastModified"),
            "collection": p.get("collectionCode") or extra.get("collection") or "CHRG",
            **extra
        })
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.drop_duplicates(subset=["packageId"])
    return df

def filter_house(df: pd.DataFrame, include_joint: bool = False) -> pd.DataFrame:
    pid = df["packageId"].astype(str)
    mask = pid.str.match(HOUSE_PID_RE)
    if include_joint:
        mask = mask | pid.str.match(JOINT_PID_RE)
    return df[mask].copy()


# =========================
# MAIN: GET ALL HOUSE HEARINGS FOR YEAR RANGE
# =========================

def fetch_chrg_by_year_slices(
    year: int,
    slice_mode: str = "month",  # "month" or "quarter"
    throttle_seconds: float = THROTTLE_SECONDS,
) -> pd.DataFrame:
    """
    Fetch ALL CHRG packages for the year by slices (month/quarter), no congress/docClass filters.
    Then dedupe by packageId.
    """
    slicer = month_slices if slice_mode.lower() == "month" else quarter_slices
    all_pkgs: List[Dict] = []
    audit = []

    with requests.Session() as session:
        for start, end, label in slicer(year):
            print(f"\n=== Year {year} slice {label} ({start}..{end}) ===")
            raw = 0
            for p in iter_published_packages(
                session=session,
                date_issued_start=start,
                date_issued_end=end,
                collection="CHRG",
                throttle_seconds=throttle_seconds,
            ):
                all_pkgs.append(p)
                raw += 1
            audit.append({"slice": label, "raw_rows": raw})

    df = packages_to_df(all_pkgs, extra={"slice_year": year, "slice_mode": slice_mode})
    print("\nSlice audit:")
    print(pd.DataFrame(audit).to_string(index=False))
    print(f"\nUnique CHRG packages (deduped): {len(df)}")
    return df


def download_house_hearings_years(
    years: List[int],
    out_dir: str = "govinfo_house_hearings",
    slice_mode: str = "month",
    include_joint: bool = False,
) -> pd.DataFrame:
    """
    For each year:
      1) fetch ALL CHRG via /published slices
      2) filter to House (hhrg) (optionally + joint)
      3) write yearly CSVs
    Finally combine all and global-dedupe by packageId.
    """
    os.makedirs(out_dir, exist_ok=True)
    failed = []

    for y in years:
        out_all = os.path.join(out_dir, f"CHRG_all_{y}_{slice_mode}.csv")
        out_house = os.path.join(out_dir, f"CHRG_house_{y}_{slice_mode}{'_plus_joint' if include_joint else ''}.csv")

        # Resume-friendly
        if os.path.exists(out_house):
            print(f"Skip existing: {out_house}")
            continue

        try:
            df_all = fetch_chrg_by_year_slices(y, slice_mode=slice_mode)
            df_house = filter_house(df_all, include_joint=include_joint)

            # Save both (helps debugging what’s filtered out)
            df_all.to_csv(out_all, index=False)
            df_house.to_csv(out_house, index=False)

            print(f"Saved ALL:   {out_all}   (rows={len(df_all)})")
            print(f"Saved HOUSE: {out_house} (rows={len(df_house)})")

        except Exception as e:
            print(f"FAILED year={y}: {e}")
            failed.append({"year": y, "error": str(e)})

    if failed:
        pd.DataFrame(failed).to_csv(os.path.join(out_dir, "failed_years.csv"), index=False)

    # Combine all house files
    house_files = [
        os.path.join(out_dir, f)
        for f in os.listdir(out_dir)
        if f.startswith("CHRG_house_") and f.endswith(".csv")
    ]
    if not house_files:
        return pd.DataFrame()

    all_house = pd.concat((pd.read_csv(f) for f in house_files), ignore_index=True)
    all_house = all_house.drop_duplicates(subset=["packageId"])
    assert all_house["packageId"].duplicated().sum() == 0
    return all_house


# =========================
# OPTIONAL: INCREMENTAL UPDATES (LAST MODIFIED)
# =========================

def fetch_recent_chrg_updates(
    last_modified_start: str,
    last_modified_end: Optional[str] = None,
    include_joint: bool = False,
) -> pd.DataFrame:
    """
    Use /collections/CHRG/{start}/{end?} to catch items added/modified later,
    even if their dateIssued is older. :contentReference[oaicite:9]{index=9}
    """
    pkgs = []
    with requests.Session() as session:
        for p in iter_collection_modified_packages(
            session=session,
            collection="CHRG",
            last_modified_start=last_modified_start,
            last_modified_end=last_modified_end,
        ):
            pkgs.append(p)

    df = packages_to_df(pkgs, extra={"source": "collections_lastModified"})
    df_house = filter_house(df, include_joint=include_joint)
    return df_house

In [11]:
# =========================
# EXAMPLE NOTEBOOK USAGE
# =========================

# 1) Set your key in-notebook if needed:
# import os
# os.environ["GOVINFO_API_KEY"] = "YOUR_KEY"
# API_KEY = os.getenv("GOVINFO_API_KEY")

# 2) Test one year first:
df_house_2025 = download_house_hearings_years([2025], slice_mode="month", include_joint=False)
df_house_2025.head()



Skip existing: govinfo_house_hearings\CHRG_house_2025_month.csv


,packageId,title,congress,dateIssued,chamber,docClass,lastModified,collection,slice_year,slice_mode
0,CHRG-114hhrg98359,Supplemental Nutrition Assistance Program,114,2016-01-12,NaN,HHRG,2025-08-26T19:47:35Z,CHRG,2016,month
1,CHRG-114hhrg99798,Opportunities and Challenges Facing the Nation...,114,2016-01-12,NaN,HHRG,2025-08-23T18:29:03Z,CHRG,2016,month
2,CHRG-114hhrg98888,[H.A.S.C. No. 114-79] National Academies Study...,114,2016-01-12,NaN,HHRG,2025-08-23T17:40:29Z,CHRG,2016,month
3,CHRG-114hhrg98282,The Original Meaning of the Origination Clause,114,2016-01-13,NaN,HHRG,2025-08-23T11:13:39Z,CHRG,2016,month
4,CHRG-114hhrg98890,[H.A.S.C. No. 114-81] Views on Commissary Reform,114,2016-01-13,NaN,HHRG,2025-08-23T10:33:25Z,CHRG,2016,month


In [4]:
# 3) Download a range of years:
years = list(range(1995, 2026))
house_all = download_house_hearings_years(years, out_dir="govinfo_house_hearings_monthly", slice_mode="month")
print("Total unique House hearings:", len(house_all))

# 4) Optional: catch updates from the last 30 days (example):
# start = (datetime.utcnow() - timedelta(days=30)).strftime("%Y-%m-%d")
# updates = fetch_recent_chrg_updates(start)
# print("Recent House updates:", len(updates))


=== Year 1995 slice 1995-01 (1995-01-01..1995-01-31) ===
[published] 1995-01-01..1995-01-31 | page 1 | offsetMark=*

=== Year 1995 slice 1995-02 (1995-02-01..1995-02-28) ===
[published] 1995-02-01..1995-02-28 | page 1 | offsetMark=*

=== Year 1995 slice 1995-03 (1995-03-01..1995-03-31) ===
[published] 1995-03-01..1995-03-31 | page 1 | offsetMark=*

=== Year 1995 slice 1995-04 (1995-04-01..1995-04-30) ===
[published] 1995-04-01..1995-04-30 | page 1 | offsetMark=*

=== Year 1995 slice 1995-05 (1995-05-01..1995-05-31) ===
[published] 1995-05-01..1995-05-31 | page 1 | offsetMark=*

=== Year 1995 slice 1995-06 (1995-06-01..1995-06-30) ===
[published] 1995-06-01..1995-06-30 | page 1 | offsetMark=*

=== Year 1995 slice 1995-07 (1995-07-01..1995-07-31) ===
[published] 1995-07-01..1995-07-31 | page 1 | offsetMark=*

=== Year 1995 slice 1995-08 (1995-08-01..1995-08-31) ===
[published] 1995-08-01..1995-08-31 | page 1 | offsetMark=*

=== Year 1995 slice 1995-09 (1995-09-01..1995-09-30) ===
[publi

In [2]:
df = house_all
df["year"] = pd.to_datetime(df["dateIssued"], errors="coerce").dt.year

HOUSE_RE  = re.compile(r"^CHRG-\d+hhrg", re.IGNORECASE)
SENATE_RE = re.compile(r"^CHRG-\d+shrg", re.IGNORECASE)
JOINT_RE  = re.compile(r"^CHRG-\d+jhrg", re.IGNORECASE)

def hearing_type(pid: str) -> str:
    if HOUSE_RE.match(pid):
        return "house"
    if SENATE_RE.match(pid) or JOINT_RE.match(pid):
        return "non_house"
    return "unknown"

df["hearing_type"] = df["packageId"].astype(str).apply(hearing_type)

summary = (
    df.groupby(["year", "congress"])
      .agg(
          total_hearings=("packageId", "nunique"),
          house_hearings=("hearing_type", lambda x: (x == "house").sum()),
          non_house_hearings=("hearing_type", lambda x: (x == "non_house").sum()),
      )
      .reset_index()
      .sort_values(["year", "congress"])
)

NameError: name 'house_all' is not defined

In [7]:
import os
import re
import pandas as pd

# ----------------------------
# Config
# ----------------------------

INPUT_DIR = "govinfo_house_hearings_monthly"
OUTPUT_CSV = "govinfo_house_and_joint_hearings.csv"

HOUSE_RE = re.compile(r"^CHRG-\d+hhrg", re.IGNORECASE)
JOINT_RE = re.compile(r"^CHRG-\d+jhrg", re.IGNORECASE)

# ----------------------------
# Load all CSVs
# ----------------------------

files = [
    os.path.join(INPUT_DIR, f)
    for f in os.listdir(INPUT_DIR)
    if f.endswith(".csv")
]

if not files:
    raise RuntimeError(f"No CSV files found in {INPUT_DIR}")

print(f"Loading {len(files)} files...")

df = pd.concat(
    (pd.read_csv(f) for f in files),
    ignore_index=True
)

print(f"Total rows loaded: {len(df)}")

# ----------------------------
# Filter: House + Joint only
# ----------------------------

pid = df["packageId"].astype(str)

mask_house = pid.str.match(HOUSE_RE)
mask_joint = pid.str.match(JOINT_RE)

df_house_joint = df[mask_house | mask_joint].copy()

print(f"Rows after House + Joint filter: {len(df_house_joint)}")

# ----------------------------
# Deduplicate by packageId
# ----------------------------

before = len(df_house_joint)
df_house_joint = df_house_joint.drop_duplicates(subset=["packageId"])
after = len(df_house_joint)

print(f"Deduplicated: {before} → {after} unique packageIds")

# ----------------------------
# Optional: sanity checks
# ----------------------------

assert df_house_joint["packageId"].duplicated().sum() == 0

# Check composition
counts = (
    df_house_joint["packageId"]
    .str.extract(r"(hhrg|jhrg)", expand=False)
    .str.lower()
    .value_counts()
)

print("\nHearing type counts:")
print(counts)


df_house_joint["year"] = (
    pd.to_datetime(df_house_joint["dateIssued"], errors="coerce")
      .dt.year
)


import re

HOUSE_RE  = re.compile(r"^CHRG-\d+hhrg", re.IGNORECASE)
JOINT_RE  = re.compile(r"^CHRG-\d+jhrg", re.IGNORECASE)
SENATE_RE = re.compile(r"^CHRG-\d+shrg", re.IGNORECASE)

def infer_chamber(package_id: str) -> str:
    if HOUSE_RE.match(package_id):
        return "House"
    if JOINT_RE.match(package_id):
        return "Joint"
    if SENATE_RE.match(package_id):
        return "Senate"
    return "Unknown"

df_house_joint["chamber"] = (
    df_house_joint["packageId"]
    .astype(str)
    .apply(infer_chamber)
)




# ----------------------------
# Save
# ----------------------------

df_house_joint.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved: {OUTPUT_CSV}")


Loading 62 files...
Total rows loaded: 55127
Rows after House + Joint filter: 41548
Deduplicated: 41548 → 21155 unique packageIds

Hearing type counts:
packageId
hhrg    20393
jhrg      762
Name: count, dtype: int64

Saved: govinfo_house_and_joint_hearings.csv


In [8]:
df_house_joint["chamber"].value_counts()


chamber
House    20393
Joint      762
Name: count, dtype: int64

# bill list

In [9]:
# Load Congress.gov API key from file
with open("congress_api_key.txt", "r") as f:
    API_KEY = f.read().strip()

if not API_KEY:
    raise RuntimeError("Congress.gov API key file is empty")

In [3]:
import os
import re
import time
import random
from datetime import datetime, timezone
from typing import Dict, List, Optional, Tuple, Iterable, Set

import requests
import pandas as pd


# =========================
# USER SETTINGS
# =========================

# Choose years to fetch
YEARS = list(range(1995, 2026))  # 1995–2025 inclusive

# Where to write month checkpoints
OUT_DIR = "congressgov_monthly_checkpoints"
os.makedirs(OUT_DIR, exist_ok=True)

# Final combined output
FINAL_CSV = "congressgov_hearings_1995_2025_with_bills_MASTER.csv"

# API config
BASE = "https://api.congress.gov/v3"
LIMIT = 250
SLEEP_BETWEEN_CALLS_SEC = 0.12
MAX_TRIES = 10
BACKOFF_CAP_SEC = 60

# Congress range covering 1995–2025
CONGRESS_START = 104  # starts 1995
CONGRESS_END = 119    # starts 2025

# Hearing list chambers to try
HEARING_CHAMBERS = ["house", "senate", "nochamber"]

# For committee meetings, chamber sometimes varies; use small fallback list
MEETING_CHAMBERS_TRY = {
    "house": ["house"],
    "senate": ["senate"],
    "nochamber": ["nochamber", "house", "senate"],
}

# If a LIST page for a given congress/chamber is repeatedly failing (like 109/senate offset=1000),
# we will SKIP that congress/chamber instead of crashing everything.
SKIP_ON_PERSISTENT_LIST_FAILURE = True

# =========================
# API KEY
# =========================

def load_api_key(path: str = "congress_api_key.txt") -> str:
    with open(path, "r") as f:
        key = f.read().strip()
    if not key:
        raise RuntimeError(f"API key file is empty: {path}")
    return key

API_KEY = load_api_key("congress_api_key.txt")


# =========================
# HTTP helpers
# =========================

session = requests.Session()
session.headers.update({"Accept": "application/json"})


def _sleep_polite():
    time.sleep(SLEEP_BETWEEN_CALLS_SEC)


def get_json(url: str, params: Optional[dict] = None, max_tries: int = MAX_TRIES) -> dict:
    """
    GET JSON with retry on 429 and 5xx. Raises RuntimeError after max_tries.
    """
    params = params or {}
    params.setdefault("format", "json")
    params.setdefault("api_key", API_KEY)

    last_status = None
    for i in range(1, max_tries + 1):
        resp = session.get(url, params=params, timeout=60)
        last_status = resp.status_code

        if last_status == 200:
            _sleep_polite()
            return resp.json()

        if last_status == 429 or (500 <= last_status < 600):
            base_wait = min(BACKOFF_CAP_SEC, 2 ** (i - 1))
            wait = base_wait + random.random()
            print(f"HTTP {last_status} (try {i}/{max_tries}) sleeping {wait:.1f}s: {resp.url}")
            time.sleep(wait)
            continue

        resp.raise_for_status()

    raise RuntimeError(f"Failed after {max_tries} tries. Last status={last_status} url={resp.url}")


def paginate_list(first_url: str, first_params: dict, data_key: str) -> List[dict]:
    """
    Collect all items across pages by following pagination.next.
    If a page fails persistently, raises RuntimeError.
    """
    out: List[dict] = []
    url = first_url
    params = dict(first_params)

    while True:
        j = get_json(url, params=params)
        items = j.get(data_key, []) or []
        out.extend(items)

        nxt = (j.get("pagination") or {}).get("next")
        if not nxt:
            break

        url = nxt
        params = {"api_key": API_KEY, "format": "json"}

    return out


# =========================
# Parsing helpers
# =========================

BILL_URL_RE = re.compile(r"/v3/bill/(\d+)/([a-z]+)/(\d+)", re.IGNORECASE)

def parse_bill_from_url(url: str) -> Optional[Tuple[int, str, int]]:
    m = BILL_URL_RE.search(url)
    if not m:
        return None
    return int(m.group(1)), m.group(2).lower(), int(m.group(3))


def bill_display(bill_type: str, bill_number: int) -> str:
    t = bill_type.lower()
    if t == "hr": return f"H.R. {bill_number}"
    if t == "s": return f"S. {bill_number}"
    if t == "hjres": return f"H.J.Res. {bill_number}"
    if t == "sjres": return f"S.J.Res. {bill_number}"
    if t == "hconres": return f"H.Con.Res. {bill_number}"
    if t == "sconres": return f"S.Con.Res. {bill_number}"
    if t == "hres": return f"H.Res. {bill_number}"
    if t == "sres": return f"S.Res. {bill_number}"
    return f"{bill_type.upper()} {bill_number}"


def safe_first_date(hearing_obj: dict) -> Optional[str]:
    dates = hearing_obj.get("dates") or []
    if isinstance(dates, list):
        for d in dates:
            if isinstance(d, dict) and d.get("date"):
                return d["date"]
            if isinstance(d, str):
                return d
    return None


def year_month_from_date(date_str: Optional[str]) -> Optional[Tuple[int, int]]:
    if not date_str:
        return None
    try:
        y = int(str(date_str)[:4])
        m = int(str(date_str)[5:7])
        return y, m
    except Exception:
        return None


# =========================
# Endpoint wrappers
# =========================

def fetch_hearing_list(congress: int, chamber: str) -> List[dict]:
    url = f"{BASE}/hearing/{congress}/{chamber}"
    params = {"limit": LIMIT, "offset": 0}
    return paginate_list(url, params, data_key="hearings")


def fetch_hearing_detail(congress: int, chamber: str, jacket_number: int) -> dict:
    url = f"{BASE}/hearing/{congress}/{chamber}/{jacket_number}"
    return get_json(url)


def fetch_committee_meeting(congress: int, chamber: str, event_id: str) -> dict:
    url = f"{BASE}/committee-meeting/{congress}/{chamber}/{event_id}"
    return get_json(url)


def fetch_bill_detail(congress: int, bill_type: str, bill_number: int) -> dict:
    url = f"{BASE}/bill/{congress}/{bill_type}/{bill_number}"
    return get_json(url)


# =========================
# Checkpoint IO helpers
# =========================

def checkpoint_path(year: int, month: int) -> str:
    return os.path.join(OUT_DIR, f"hearings_{year}_{month:02d}.csv")


def load_checkpoint(year: int, month: int) -> Optional[pd.DataFrame]:
    path = checkpoint_path(year, month)
    if os.path.exists(path):
        return pd.read_csv(path)
    return None


def save_checkpoint(df: pd.DataFrame, year: int, month: int) -> None:
    path = checkpoint_path(year, month)
    df.to_csv(path, index=False)
    print(f"Saved checkpoint: {path} (rows={len(df)})")


# =========================
# Main crawler (build monthly)
# =========================

def build_monthly_checkpoints(years: Iterable[int]) -> None:
    years_set: Set[int] = set(int(y) for y in years)
    # Pre-create all (year, month) bins we care about
    target_bins = {(y, m) for y in years_set for m in range(1, 13)}

    # In-memory buffers per month; we flush to disk as we go
    buffers: Dict[Tuple[int, int], List[dict]] = {(y, m): [] for (y, m) in target_bins}

    # We’ll cache bill detail calls (huge speedup)
    bill_cache: Dict[Tuple[int, str, int], Tuple[str, str]] = {}

    # Determine which month bins already exist on disk (resume)
    done_bins = set()
    for (y, m) in list(target_bins):
        if os.path.exists(checkpoint_path(y, m)):
            done_bins.add((y, m))

    print(f"Months requested: {len(target_bins)} | Already done (will skip): {len(done_bins)}")

    for cong in range(CONGRESS_START, CONGRESS_END + 1):
        for chamber in HEARING_CHAMBERS:
            print(f"\n=== LIST hearings: congress={cong} chamber={chamber} ===")
            try:
                hearing_list = fetch_hearing_list(cong, chamber)
                print(f"Found {len(hearing_list)} list items for {cong}-{chamber}")
            except Exception as e:
                print(f"LIST FAILED for {cong}-{chamber}: {e}")
                if SKIP_ON_PERSISTENT_LIST_FAILURE:
                    print(f"Skipping {cong}-{chamber} and continuing...")
                    continue
                else:
                    raise

            for h in hearing_list:
                jacket = h.get("jacketNumber")
                if jacket is None:
                    continue

                # hearing detail
                try:
                    hd = fetch_hearing_detail(cong, chamber, int(jacket))
                except Exception as e:
                    print(f"DETAIL FAILED {cong}-{chamber}-{jacket}: {e}")
                    continue

                hearing_obj = hd.get("hearing", {}) or {}
                title = hearing_obj.get("title") or h.get("title")
                date_first = safe_first_date(hearing_obj)
                ym = year_month_from_date(date_first)

                if ym is None:
                    continue

                y, m = ym
                if (y not in years_set):
                    continue

                # If this month was already checkpointed, skip work
                if (y, m) in done_bins:
                    continue

                assoc = hearing_obj.get("associatedMeeting") or {}
                event_id = assoc.get("eventId")

                bills_for_hearing: List[Tuple[int, str, int]] = []
                bill_displays: List[str] = []
                bill_latest_actions: List[str] = []

                if event_id:
                    meeting_obj = None
                    for meet_ch in MEETING_CHAMBERS_TRY.get(chamber, ["nochamber", "house", "senate"]):
                        try:
                            cm = fetch_committee_meeting(cong, meet_ch, str(event_id))
                            meeting_obj = cm.get("committeeMeeting") or cm.get("committee_meeting") or {}
                            if meeting_obj:
                                break
                        except Exception:
                            continue

                    if meeting_obj:
                        related = meeting_obj.get("relatedItems") or meeting_obj.get("related_items") or []
                        if isinstance(related, dict):
                            related_items = related.get("item") or related.get("items") or []
                        else:
                            related_items = related

                        bill_urls = []
                        for it in related_items or []:
                            if isinstance(it, dict):
                                u = it.get("url") or it.get("URL")
                                if u and "/bill/" in u:
                                    bill_urls.append(u)

                        for u in bill_urls:
                            parsed = parse_bill_from_url(u)
                            if parsed:
                                bills_for_hearing.append(parsed)

                bills_for_hearing = list(dict.fromkeys(bills_for_hearing))

                for (bcong, btype, bnum) in bills_for_hearing:
                    key = (bcong, btype, bnum)
                    if key not in bill_cache:
                        disp = bill_display(btype, bnum)
                        latest_str = ""
                        try:
                            bd = fetch_bill_detail(bcong, btype, bnum)
                            bill_obj = bd.get("bill", {}) or {}
                            latest = bill_obj.get("latestAction") or {}
                            ad = latest.get("actionDate") or ""
                            txt = latest.get("text") or ""
                            latest_str = (ad + " — " + txt).strip(" —")
                        except Exception:
                            pass
                        bill_cache[key] = (disp, latest_str)

                    disp, latest_str = bill_cache[key]
                    bill_displays.append(disp)
                    if latest_str:
                        bill_latest_actions.append(latest_str)

                has_bill = 1 if bill_displays else 0

                buffers[(y, m)].append(
                    {
                        "hearing_key": f"{cong}-{chamber}-{int(jacket)}",
                        "congress": cong,
                        "chamber": chamber,
                        "jacketNumber": int(jacket),
                        "title": title,
                        "date_first": date_first,
                        "year": y,
                        "month": m,
                        "eventId": event_id,
                        "has_bill": has_bill,
                        "bill_numbers": "; ".join(bill_displays) if bill_displays else "",
                        "bill_latest_actions": "; ".join(bill_latest_actions) if bill_latest_actions else "",
                        "update_timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
                    }
                )

                # Periodically flush this month to disk so you don't lose progress.
                # Flush when buffer gets "big enough"
                if len(buffers[(y, m)]) >= 2000:
                    df_chunk = pd.DataFrame(buffers[(y, m)])
                    # If a checkpoint already exists (partial runs), append-dedupe
                    existing = load_checkpoint(y, m)
                    if existing is not None and len(existing):
                        df_chunk = pd.concat([existing, df_chunk], ignore_index=True)
                        df_chunk = df_chunk.drop_duplicates(subset=["hearing_key"])
                    save_checkpoint(df_chunk, y, m)
                    buffers[(y, m)] = []

    # Final flush for all months not done
    for (y, m), rows in buffers.items():
        if (y, m) in done_bins:
            continue
        if not rows:
            # still write an empty file so it counts as done
            save_checkpoint(pd.DataFrame(columns=[
                "hearing_key","congress","chamber","jacketNumber","title","date_first","year","month",
                "eventId","has_bill","bill_numbers","bill_latest_actions","update_timestamp_utc"
            ]), y, m)
            continue

        df_month = pd.DataFrame(rows)
        existing = load_checkpoint(y, m)
        if existing is not None and len(existing):
            df_month = pd.concat([existing, df_month], ignore_index=True)
            df_month = df_month.drop_duplicates(subset=["hearing_key"])
        save_checkpoint(df_month, y, m)

    print("\nMonthly checkpointing complete.")


def combine_monthly_to_master(years: Iterable[int], out_csv: str = FINAL_CSV) -> pd.DataFrame:
    years_set = set(int(y) for y in years)
    files = []
    for y in sorted(years_set):
        for m in range(1, 13):
            p = checkpoint_path(y, m)
            if os.path.exists(p):
                files.append(p)

    if not files:
        raise RuntimeError(f"No checkpoint files found in {OUT_DIR}")

    print(f"Combining {len(files)} monthly files...")
    df = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)
    if not df.empty:
        df = df.drop_duplicates(subset=["hearing_key"])
        df = df.sort_values(["year", "month", "congress", "chamber", "jacketNumber"]).reset_index(drop=True)

    df.to_csv(out_csv, index=False)
    print(f"Saved MASTER: {out_csv} (rows={len(df)})")
    return df


# =========================
# RUN (Jupyter-friendly)
# =========================

# 1) Build/Resume monthly checkpoints
build_monthly_checkpoints(YEARS)

# 2) Combine to a single master CSV
master = combine_monthly_to_master(YEARS, FINAL_CSV)

# Quick peek
master.head()


Months requested: 372 | Already done (will skip): 372

=== LIST hearings: congress=104 chamber=house ===
Found 284 list items for 104-house

=== LIST hearings: congress=104 chamber=senate ===
Found 0 list items for 104-senate

=== LIST hearings: congress=104 chamber=nochamber ===
Found 2 list items for 104-nochamber

=== LIST hearings: congress=105 chamber=house ===
Found 314 list items for 105-house

=== LIST hearings: congress=105 chamber=senate ===
Found 99 list items for 105-senate

=== LIST hearings: congress=105 chamber=nochamber ===
Found 2 list items for 105-nochamber

=== LIST hearings: congress=106 chamber=house ===
Found 1062 list items for 106-house

=== LIST hearings: congress=106 chamber=senate ===
Found 618 list items for 106-senate
DETAIL FAILED 106-senate-57907: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

=== LIST hearings: congress=106 chamber=nochamber ===
Found 12 list items for 106-nochamber

=== LIST hearings: cong

,hearing_key,congress,chamber,jacketNumber,title,date_first,year,month,eventId,has_bill,bill_numbers,bill_latest_actions,update_timestamp_utc
0,104-house-20049,104,house,20049,CONTRACT WITH AMERICA-SAVINGS AND INVESTMENT,1995-01-24,1995,1,NaN,0,NaN,NaN,2026-01-15T08:36:14+00:00
1,104-house-23219,104,house,23219,OVERSIGHT HEARING ON THE DEPARTMENT OF LABOR,1995-01-01,1995,1,NaN,0,NaN,NaN,2026-01-15T08:35:10+00:00
2,104-house-88698,104,house,88698,THE REGULATORY TRANSITION ACT OF 1995,1995-01-19,1995,1,NaN,0,NaN,NaN,2026-01-15T08:35:04+00:00
3,104-house-88880,104,house,88880,SOCIAL SECURITY EARNINGS LIMIT PROVISION OF TH...,1995-01-09,1995,1,NaN,0,NaN,NaN,2026-01-15T08:36:15+00:00
4,104-house-88881,104,house,88881,INTRODUCING GRAY WOLVES IN YELLOWSTONE AND IDAHO,1995-01-26,1995,1,NaN,0,NaN,NaN,2026-01-15T08:33:54+00:00


In [2]:
bills_df = pd.read_csv("congressgov_HEARINGS_from_committee_meetings_1995_2025.csv")

In [3]:
bills_df

,eventId,congress,api_chamber,meeting_type,date,year,month,title,committee,has_bill,bill_numbers,bill_latest_actions,update_timestamp_utc
0,100565,111,house,Hearing,2009-02-25T15:00:00Z,2009,2,“DHS: The Path Forward.”,NaN,0,NaN,NaN,2026-01-20T11:35:11+00:00
1,100566,111,house,Hearing,2009-03-04T19:00:00Z,2009,3,“Examining 287(g): The Role of State and Local...,NaN,0,NaN,NaN,2026-01-20T11:35:12+00:00
2,100033,112,house,Meeting,2011-01-26T15:00:00Z,2011,1,Organizational Meeting,NaN,0,NaN,NaN,2026-01-20T11:35:29+00:00
3,103896,112,house,Markup,2011-01-26T14:00:00Z,2011,1,House Budget Committee Organizational Meeting,NaN,0,NaN,NaN,2026-01-20T11:35:35+00:00
4,103897,112,house,Hearing,2011-01-26T15:00:00Z,2011,1,The Fiscal Consequences of the New Health Care...,NaN,0,NaN,NaN,2026-01-20T11:35:35+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...
14919,337707,119,senate,Meeting,2025-12-02T19:30:00Z,2025,12,Hearings to examine China's challenge to Ameri...,NaN,0,NaN,NaN,2026-01-20T15:30:05+00:00
14920,337700,119,senate,Meeting,2025-12-02T15:00:00Z,2025,12,Hearings to examine America's communications n...,NaN,0,NaN,NaN,2026-01-20T15:30:05+00:00
14921,337706,119,senate,Meeting,2025-12-03T20:00:00Z,2025,12,To receive a closed briefing on certain intell...,NaN,0,NaN,NaN,2026-01-20T15:30:06+00:00
14922,337705,119,senate,Meeting,2025-12-02T20:00:00Z,2025,12,To receive a closed briefing on certain intell...,NaN,0,NaN,NaN,2026-01-20T15:30:06+00:00


In [5]:
bills_df

,eventId,congress,api_chamber,meeting_type,date,year,month,title,committee,has_bill,bill_numbers,bill_latest_actions,update_timestamp_utc
0,100565,111,house,Hearing,2009-02-25T15:00:00Z,2009,2,“DHS: The Path Forward.”,NaN,0,NaN,NaN,2026-01-20T11:35:11+00:00
1,100566,111,house,Hearing,2009-03-04T19:00:00Z,2009,3,“Examining 287(g): The Role of State and Local...,NaN,0,NaN,NaN,2026-01-20T11:35:12+00:00
2,100033,112,house,Meeting,2011-01-26T15:00:00Z,2011,1,Organizational Meeting,NaN,0,NaN,NaN,2026-01-20T11:35:29+00:00
3,103896,112,house,Markup,2011-01-26T14:00:00Z,2011,1,House Budget Committee Organizational Meeting,NaN,0,NaN,NaN,2026-01-20T11:35:35+00:00
4,103897,112,house,Hearing,2011-01-26T15:00:00Z,2011,1,The Fiscal Consequences of the New Health Care...,NaN,0,NaN,NaN,2026-01-20T11:35:35+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...
14919,337707,119,senate,Meeting,2025-12-02T19:30:00Z,2025,12,Hearings to examine China's challenge to Ameri...,NaN,0,NaN,NaN,2026-01-20T15:30:05+00:00
14920,337700,119,senate,Meeting,2025-12-02T15:00:00Z,2025,12,Hearings to examine America's communications n...,NaN,0,NaN,NaN,2026-01-20T15:30:05+00:00
14921,337706,119,senate,Meeting,2025-12-03T20:00:00Z,2025,12,To receive a closed briefing on certain intell...,NaN,0,NaN,NaN,2026-01-20T15:30:06+00:00
14922,337705,119,senate,Meeting,2025-12-02T20:00:00Z,2025,12,To receive a closed briefing on certain intell...,NaN,0,NaN,NaN,2026-01-20T15:30:06+00:00


In [6]:
bills_df["has_bill"] = bills_df["has_bill"].astype(int)

# --- rate per year ---
rate_by_year = (
    bills_df
    .groupby("year")["has_bill"]
    .agg(rate="mean", n="size", n_with_bill="sum")
    .reset_index()
)

print(rate_by_year)

    year      rate     n  n_with_bill
0   2009  0.000000     2            0
1   2011  0.111111    63            7
2   2012  0.078431    51            4
3   2013  0.202683   671          136
4   2014  0.200000   570          114
5   2015  0.194444  1152          224
6   2016  0.197674   602          119
7   2017  0.233253  1239          289
8   2018  0.215876  1033          223
9   2019  0.166228  1901          316
10  2020  0.158650  1185          188
11  2021  0.121212   858          104
12  2022  0.175182   685          120
13  2023  0.173637  1889          328
14  2024  0.189114  1433          271
15  2025  0.156604  1590          249


In [11]:
import time, random
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

BASE = "https://www.congress.gov"

def make_session() -> requests.Session:
    s = requests.Session()

    # retries for transient issues (NOT for 403, but helpful generally)
    retry = Retry(
        total=8,
        backoff_factor=0.8,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset(["GET"]),
        raise_on_status=False,
    )
    s.mount("https://", HTTPAdapter(max_retries=retry))

    # Browser-ish headers
    s.headers.update({
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/122.0.0.0 Safari/537.36"
        ),
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
        "Accept-Encoding": "gzip, deflate, br",
        "Connection": "keep-alive",
        "Upgrade-Insecure-Requests": "1",
        "Sec-Fetch-Dest": "document",
        "Sec-Fetch-Mode": "navigate",
        "Sec-Fetch-Site": "same-origin",
        "Sec-Fetch-User": "?1",
        "Cache-Control": "no-cache",
        "Pragma": "no-cache",
        # some WAFs like having a referer
        "Referer": BASE + "/",
    })

    return s

def warm_up(sess: requests.Session):
    # hit homepage to get cookies
    r = sess.get(BASE + "/", timeout=60)
    polite_sleep()
    return r.status_code


def polite_sleep(a=0.7, b=1.6):
    time.sleep(random.uniform(a, b))

def fetch_html(sess: requests.Session, url: str, tries: int = 6) -> str:
    last_exc = None
    last_status = None

    for i in range(1, tries + 1):
        try:
            r = sess.get(url, timeout=60)
            last_status = r.status_code

            if r.status_code == 200:
                return r.text

            if r.status_code == 403:
                # re-warm + backoff
                polite_sleep(1.5, 3.5)
                try:
                    sess.get("https://www.congress.gov/", timeout=60)
                except Exception:
                    pass
                polite_sleep(1.5, 3.5)
                continue

            # other statuses
            polite_sleep(1.0 * i, 2.0 * i)

        except requests.exceptions.RequestException as e:
            last_exc = e
            # backoff
            polite_sleep(1.0 * i, 2.0 * i)

            # sometimes a fresh session helps (proxy/VPN hiccups)
            if i in (3, 5):
                try:
                    sess.close()
                except Exception:
                    pass
                sess = make_session()
            continue

    msg = f"Failed to fetch {url}. last_status={last_status} last_exc={repr(last_exc)}"
    raise RuntimeError(msg)


In [12]:
import re
import pandas as pd
from bs4 import BeautifulSoup
from typing import List, Dict, Optional, Tuple

BILL_URL_RE = re.compile(r"/bill/(\d+)/(hr|s|hjres|sjres|hconres|sconres|hres|sres)/(\d+)", re.I)
BILL_TEXT_RE = re.compile(
    r"\b(?P<prefix>H\.?R\.?|S\.?|H\.?J\.?RES\.?|S\.?J\.?RES\.?|H\.?CON\.?RES\.?|S\.?CON\.?RES\.?|H\.?RES\.?|S\.?RES\.?)\s*\.?\s*(?P<num>\d+)\b",
    re.I
)

def normalize_bill_type(prefix: str) -> Optional[str]:
    p = re.sub(r"[^A-Z]", "", prefix.upper())
    return {
        "HR": "hr", "S": "s",
        "HJRES": "hjres", "SJRES": "sjres",
        "HCONRES": "hconres", "SCONRES": "sconres",
        "HRES": "hres", "SRES": "sres",
    }.get(p)

def bill_display(bill_type: str, bill_number: int) -> str:
    t = bill_type.lower()
    if t == "hr": return f"H.R. {bill_number}"
    if t == "s": return f"S. {bill_number}"
    if t == "hjres": return f"H.J.Res. {bill_number}"
    if t == "sjres": return f"S.J.Res. {bill_number}"
    if t == "hconres": return f"H.Con.Res. {bill_number}"
    if t == "sconres": return f"S.Con.Res. {bill_number}"
    if t == "hres": return f"H.Res. {bill_number}"
    if t == "sres": return f"S.Res. {bill_number}"
    return f"{bill_type.upper()} {bill_number}"

def parse_bill_from_url(href: str) -> Optional[Tuple[int, str, int]]:
    m = BILL_URL_RE.search(href or "")
    if not m:
        return None
    return int(m.group(1)), m.group(2).lower(), int(m.group(3))

def find_results_table(soup: BeautifulSoup):
    # Find the table whose header row contains "Event ID" and "Hearing Date"
    for table in soup.find_all("table"):
        headers = [th.get_text(" ", strip=True).lower() for th in table.find_all("th")]
        if any("event id" in h for h in headers) and any("hearing date" in h for h in headers):
            return table
    return None

def extract_bills_from_cell(cell, congress: int) -> List[Tuple[int, str, int]]:
    found: List[Tuple[int, str, int]] = []

    for a in cell.select("a[href]"):
        parsed = parse_bill_from_url(a["href"])
        if parsed:
            found.append(parsed)

    txt = cell.get_text(" ", strip=True)
    for m in BILL_TEXT_RE.finditer(txt):
        btype = normalize_bill_type(m.group("prefix"))
        if btype:
            found.append((congress, btype, int(m.group("num"))))

    # de-dupe
    out, seen = [], set()
    for x in found:
        if x not in seen:
            out.append(x); seen.add(x)
    return out

def scrape_house_transcripts_congress(congress: int, sess: requests.Session) -> pd.DataFrame:
    url = f"{BASE}/house-hearing-transcripts/{congress}th-congress"
    html = fetch_html(sess, url)

    soup = BeautifulSoup(html, "html.parser")
    table = find_results_table(soup)
    if table is None:
        raise RuntimeError("Could not locate the transcript results table (page layout changed?).")

    rows_out: List[Dict] = []
    trs = table.find_all("tr")

    for tr in trs[1:]:
        tds = tr.find_all("td")
        if len(tds) < 4:
            continue

        event_txt = tds[0].get_text(" ", strip=True)
        date_txt = tds[1].get_text(" ", strip=True)
        title_txt = tds[2].get_text(" ", strip=True)
        committee_txt = tds[3].get_text(" ", strip=True)

        m = re.search(r"\bEvent\s+(\d+)\b", event_txt, re.I)
        if not m:
            continue
        event_id = int(m.group(1))

        bills: List[Tuple[int, str, int]] = []
        if len(tds) >= 5:
            bills.extend(extract_bills_from_cell(tds[4], congress))

        # fallback: scan title for bill citations
        for mm in BILL_TEXT_RE.finditer(title_txt):
            btype = normalize_bill_type(mm.group("prefix"))
            if btype:
                bills.append((congress, btype, int(mm.group("num"))))

        # de-dupe
        bills2, seen = [], set()
        for b in bills:
            if b not in seen:
                bills2.append(b); seen.add(b)

        bill_displays = [bill_display(bt, bn) for _, bt, bn in bills2]
        rows_out.append({
            "event_id": event_id,
            "congress": congress,
            "date": date_txt,
            "title": title_txt,
            "committee": committee_txt,
            "has_bill": 1 if bill_displays else 0,
            "bill_numbers": "; ".join(bill_displays) if bill_displays else "",
            "source_url": url,
        })

    df = pd.DataFrame(rows_out).drop_duplicates(subset=["event_id"]).reset_index(drop=True)
    return df


In [13]:
sess = make_session()
print("Warm-up status:", warm_up(sess))

dfs = []
for c in [108, 109]:
    print("Scraping", c)
    dfs.append(scrape_house_transcripts_congress(c, sess))
    polite_sleep()

out = pd.concat(dfs, ignore_index=True)
out.to_csv("house_hearing_transcripts_sample.csv", index=False)

print(out.groupby("congress")["has_bill"].agg(["mean","size","sum"]))


Warm-up status: 403
Scraping 108


RuntimeError: Failed to fetch https://www.congress.gov/house-hearing-transcripts/108th-congress. last_status=403 last_exc=None

In [14]:
import requests
r = requests.get("https://www.congress.gov/house-hearing-transcripts/108th-congress", timeout=30)
print(r.status_code)
print(r.headers.get("server"))
print(r.text[:200])


403
cloudflare
<!DOCTYPE html><html lang="en-US"><head><title>Just a moment...</title><meta http-equiv="Content-Type" content="text/html; charset=UTF-8"><meta http-equiv="X-UA-Compatible" content="IE=Edge"><meta nam


In [16]:
path = r"C:\Users\USER\Downloads\bicam_hearings\hearings_texts.csv"
df = pd.read_csv(path)

print(df.shape)
print(df.head())

(29196, 4)
      hearing_id raw_text                                                pdf  \
0  hhrg44535-117      NaN  https://congress.gov/117/chrg/CHRG-117hhrg4453...   
1  hhrg44535-117      NaN  https://congress.gov/117/chrg/CHRG-117hhrg4453...   
2  hhrg48808-117      NaN  https://congress.gov/117/chrg/CHRG-117hhrg4880...   
3  hhrg44535-117      NaN  https://congress.gov/117/chrg/CHRG-117hhrg4453...   
4  hhrg44535-117      NaN  https://congress.gov/117/chrg/CHRG-117hhrg4453...   

                                      formatted_text  
0  https://congress.gov/117/chrg/CHRG-117hhrg4453...  
1  https://congress.gov/117/chrg/CHRG-117hhrg4453...  
2  https://congress.gov/117/chrg/CHRG-117hhrg4880...  
3  https://congress.gov/117/chrg/CHRG-117hhrg4453...  
4  https://congress.gov/117/chrg/CHRG-117hhrg4453...  


In [53]:
# --- 1) Infer congress + chamber from hearing_id -----------------------------
# congress is the trailing digits after the last "-"
df = df.copy()
df["congress"] = df["hearing_id"].str.extract(r"-(\d+)$").astype("Int64")

# infer chamber from the prefix (common patterns: hhrg = House hearing, shrg = Senate hearing)
df["chamber"] = (
    df["hearing_id"]
      .str.lower()
      .str.extract(r"^([a-z]+)")
      .iloc[:, 0]
      .map({"hhrg": "House", "shrg": "Senate"})
      .fillna("Unknown")
)

# --- 2) Define "has text" at the row level ----------------------------------
# Treat NaN/None and empty/whitespace-only as missing
row_has_text = (
    df["raw_text"].notna()
    & df["raw_text"].astype(str).str.strip().ne("")
)

# --- 3) Collapse to one row per hearing (per congress, per chamber) ----------
hearing_level = (
    df.assign(row_has_text=row_has_text)
      .groupby(["congress", "chamber", "hearing_id"], dropna=False)["row_has_text"]
      .any()  # a hearing "has text" if ANY of its rows has text
      .reset_index(name="has_text")
)

# --- 4) Summarize per congress x chamber ------------------------------------
summary_missing_text = (
    hearing_level
      .groupby(["congress", "chamber"], dropna=False)
      .agg(
          n_hearings=("hearing_id", "nunique"),
          n_with_text=("has_text", "sum"),
      )
      .assign(
          n_missing=lambda d: d["n_hearings"] - d["n_with_text"],
          missing_rate=lambda d: d["n_missing"] / d["n_hearings"],
      )
      .reset_index()
      .sort_values(["congress", "chamber"])
)

# Optional: nicer display (percent + rounding)
summary_missing_text["missing_rate_pct"] = (summary_missing_text["missing_rate"] * 100).round(2)
summary_missing_text = summary_missing_text.drop(columns=["missing_rate"])

summary_missing_text[summary_missing_text['chamber'].str.lower() == 'house']

,congress,chamber,n_hearings,n_with_text,n_missing,missing_rate_pct
0,104,House,197,192,5,2.54
2,105,House,177,175,2,1.13
5,106,House,833,713,120,14.41
8,107,House,841,733,108,12.84
11,108,House,947,901,46,4.86
14,109,House,1165,1151,14,1.20
17,110,House,1810,1753,57,3.15
20,111,House,1540,1447,93,6.04
23,112,House,1993,1908,85,4.26
26,113,House,1725,1642,83,4.81


In [35]:
result = (
    df_meta
    .groupby(['congress', 'chamber'])['hearing_id']
    .nunique()
    .reset_index(name='unique_hearing_count')
)

house = result[result['chamber'].str.lower() == 'house']

In [36]:
with pd.option_context('display.max_rows', None):
    print(house)


    congress chamber  unique_hearing_count
1         82   house                     1
3         84   house                     9
5         85   house                     5
7         86   house                     1
8         87   house                   318
11        88   house                   299
14        89   house                   318
17        90   house                   326
20        91   house                   397
23        92   house                   398
26        93   house                   571
29        94   house                   691
32        95   house                   776
35        96   house                   667
38        97   house                    77
43       103   house                   188
44       104   house                   382
48       105   house                   635
52       106   house                  1205
55       107   house                  1258
58       108   house                  1266
62       109   house                  1649
66       11